# Predicting Diabetes in Pima women: getML for simple datasets

This notebook demonstrates how getML can be applied to datasets that primarily contain one-to-one relationships, where complex feature engineering is unnecessary. Without employing its feature learning arsenal, getML can also tackle these simple problems.

In many real-world scenarios, datasets involve a variety of relationships, including one-to-many, many-to-one, and many-to-many. The advanced feature learning algorithms offered by getML extracts features from these datasets, which can then be used to train ML models. However, even for simple datasets with just one-to-one relationships, getML is an effective and robust tool.

_Disclaimer: This notebook is intended for educational purpose, only to help the reader get a grasp on the getml API. We do not benchmark the results against other tools._



### Dataset Description
>
> The *Pima* dataset is a well-known dataset used for medical research, specifically focusing on diabetes prediction among Pima Indian women. It originates from a study conducted by the National Institute of Diabetes and Digestive and Kidney Diseases.
> 
> **Data Model:**
> - The dataset is organized into multiple tables: `age`, `bmi`, `diastolic`, `numPreg`, `pedigree`, `plasma`, `serum`, `tricepts`, and `pima`.
> - Each table contains two columns: `arg1` (varchar) and `arg2` (decimal for most tables, varchar for `pima`).
> 
> **Task and Target Column:**
> - The primary task is *classification*, aiming to predict the presence of diabetes.
> - The target column is `arg2` in the `pima` table.
> 
> **Column Types:**
> - The dataset includes:
>   - *Varchar* (e.g., `arg1`)
>   - *Decimal* (e.g., `arg2` in most tables)
> 
> **Metadata:**
> - Size: 700 KB
> - Number of tables: 9
> - Total number of rows: 6,912
> - Total number of columns: 18
> - Missing values: No
> - Target table: `pima`
> - Target ID: `arg1`
> 
> **Research and Usage:**
> - The dataset is widely used in medical research for developing models to predict diabetes.
> - It is a popular choice for testing classification algorithms due to its real-world medical data.
> 
> This dataset provides a valuable resource for exploring medical data analysis, particularly in predicting diabetes based on various health indicators.

### Tables
Population table: pima

<h4>
  <details open>
     <summary>ER Diagram</summary>
       <img src="https://relational.fel.cvut.cz/assets/img/datasets-generated/Pima.svg" alt="Pima ER Diagram">
   </details>
</h4>

## 1. Setup

> <span style="font-weight: 500; color: #3b3b3b;">ⓘ️&nbsp; Note</span>
>
> We have prepared an environment with all necessary dependencies for you. Follow the instructions on the [README page of our `getml-relbench` repository](https://github.com/getml/getml-relbench/blob/main/ctu/README.md#getting-started) page to set it up.
>

In this step, we:
- Import necessary libraries
- Set a getML project

In [29]:
import getml
from ctu.utils.data import load_ctu_dataset

# Enable textual output to avoid rendering issues in certain JupyterLab environments
getml.utilities.progress.FORCE_TEXTUAL_OUTPUT = True
getml.utilities.progress.FORCE_MONOCHROME_OUTPUT = True

getml.set_project("pima")

  Loading pipelines... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00


Connected to project 'pima'.

---
## 2. Loading Data into getML, Inspecting, and Setting Roles (Data Annotation)


### Objectives:
- Download the data
- Inspect columns to understand the schema
- Assign appropriate roles (join_key, numerical, categorical, etc.)

In [30]:
pima, peripheral = load_ctu_dataset("Pima")

(
    age,
    bmi,
    diastolic,
    numPreg,
    pedigree,
    plasma,
    serum,
    tricepts,
) = peripheral.values()

num_preg = numPreg.with_name("num_preg").to_df("num_preg")
peripheral["num_preg"] = num_preg

Analyzing schema:   0%|          | 0/9 [00:00<?, ?it/s]

Building data:   0%|          | 0/9 [00:00<?, ?it/s]

### Setting the roles
[`Roles`](https://getml.com/latest/user_guide/concepts/annotating_data/#annotating-data-roles) allow getML to:
- Efficiently encode links between tables (`join_key`)
- Encode numerical, and categorical features (`numerical`, `categorical`)

#### Population table: pima
- `arg1` is the join key
- `arg2` is the target: 1 indicates the presence of diabetes while 0 indicates the absence
- `split` is the segregation of population of into train and validation sets

In [31]:
pima.set_role("arg1", getml.data.roles.join_key)
pima.set_role("arg2", getml.data.roles.target)
pima

name,arg1,arg2,split
role,join_key,target,unused_string
0,A1,0,train
1,A10,0,train
2,A100,0,val
3,A101,0,train
4,A102,1,train
,...,...,...
763,A95,1,train
764,A96,1,train
765,A97,1,val


#### Peripheral tables

- age: age of the person (years)
- bmi: body mass index (weight in kg / height in m²)
- diastolic: diastolic blood pressure (mm Hg)
- num_preg: number of times pregnant
- pedigree: diabetes likelihood based on family history
- plasma: plasma glucose concentration
- serum: 2-hour serum insulin (mu U/ml)
- tricepts: triceps skin fold thickness (mm)

In all of the above peripheral tables:
- `arg1` is the join key
- `arg2` is the numerical value of the quantity

In [32]:
age.set_role("arg1", getml.data.roles.join_key)
age.set_role("arg2", getml.data.roles.numerical)
age

name,arg1,arg2
role,join_key,numerical
0,A1,50
1,A10,54
2,A100,31
3,A101,33
4,A102,22
,...,...
763,A95,21
764,A96,40
765,A97,24


In [33]:
bmi.set_role("arg1", getml.data.roles.join_key)
bmi.set_role("arg2", getml.data.roles.numerical)
bmi

name,arg1,arg2
role,join_key,numerical
0,A1,33.6
1,A10,0
2,A100,49.7
3,A101,39
4,A102,26.1
,...,...
763,A95,24.7
764,A96,33.9
765,A97,31.6


In [34]:
diastolic.set_role("arg1", getml.data.roles.join_key)
diastolic.set_role("arg2", getml.data.roles.numerical)
diastolic

name,arg1,arg2
role,join_key,numerical
0,A1,72
1,A10,96
2,A100,90
3,A101,72
4,A102,60
,...,...
763,A95,82
764,A96,72
765,A97,62


In [35]:
num_preg.set_role("arg1", getml.data.roles.join_key)
num_preg.set_role("arg2", getml.data.roles.numerical)
num_preg

name,arg1,arg2
role,join_key,numerical
0,A1,6
1,A10,8
2,A100,1
3,A101,1
4,A102,1
,...,...
763,A95,2
764,A96,6
765,A97,2


In [36]:
pedigree.set_role("arg1", getml.data.roles.join_key)
pedigree.set_role("arg2", getml.data.roles.numerical)
pedigree

name,arg1,arg2
role,join_key,numerical
0,A1,0.627
1,A10,0.232
2,A100,0.325
3,A101,1.222
4,A102,0.179
,...,...
763,A95,0.761
764,A96,0.255
765,A97,0.13


In [37]:
plasma.set_role("arg1", getml.data.roles.join_key)
plasma.set_role("arg2", getml.data.roles.numerical)
plasma

name,arg1,arg2
role,join_key,numerical
0,A1,148
1,A10,125
2,A100,122
3,A101,163
4,A102,151
,...,...
763,A95,142
764,A96,144
765,A97,92


In [38]:
serum.set_role("arg1", getml.data.roles.join_key)
serum.set_role("arg2", getml.data.roles.numerical)
serum

name,arg1,arg2
role,join_key,numerical
0,A1,0
1,A10,0
2,A100,220
3,A101,0
4,A102,0
,...,...
763,A95,64
764,A96,228
765,A97,0


In [39]:
tricepts.set_role("arg1", getml.data.roles.join_key)
tricepts.set_role("arg2", getml.data.roles.numerical)
tricepts

name,arg1,arg2
role,join_key,numerical
0,A1,35
1,A10,0
2,A100,51
3,A101,0
4,A102,0
,...,...
763,A95,18
764,A96,27
765,A97,28


---
## 3. Defining the getML DataModel

The next step is to define the data model. In getML, the [`DataModel`](https://getml.com/latest/user_guide/concepts/data_model/) serves as the foundation for our ML pipelines. It informs the pipeline how dataframes are linked with each other. 

In case of feature learning, it defines the universe of possible feature paths that getML’s algorithms can explore and learn from. The data model is structured as a Directed Acyclic Graph, where each [join path](https://getml.com/latest/user_guide/concepts/data_model/#joins) represents a potential feature learning route. 

#### Key Concepts:
* The [`DataModel`](https://getml.com/latest/user_guide/concepts/data_model/) is abstract. It defines the relational structure but does not contain any data itself.
* [`Placeholders`](https://getml.com/latest/user_guide/concepts/data_model/#placeholders) represent tables in the model. These placeholders mirror the schema of actual DataFrames but are decoupled from raw data until the point of training.

The data model for `pima` dataset is simple as all the tables are joined with a simple one-to-one relationship.

In [40]:
dm = getml.data.DataModel(population=pima.to_placeholder())
dm.add(getml.data.to_placeholder(**peripheral))

dm.population.join(dm.age, on="arg1", relationship=getml.data.relationship.one_to_one)
dm.population.join(dm.bmi, on="arg1", relationship=getml.data.relationship.one_to_one)
dm.population.join(dm.diastolic, on="arg1", relationship=getml.data.relationship.one_to_one)
dm.population.join(dm.num_preg, on="arg1", relationship=getml.data.relationship.one_to_one)
dm.population.join(dm.pedigree, on="arg1", relationship=getml.data.relationship.one_to_one)
dm.population.join(dm.plasma, on="arg1", relationship=getml.data.relationship.one_to_one)
dm.population.join(dm.serum, on="arg1", relationship=getml.data.relationship.one_to_one)
dm.population.join(dm.tricepts, on="arg1", relationship=getml.data.relationship.one_to_one)

dm

,data frames,staging table
0,"pima, age, bmi, diastolic, num_preg, pedigree, plasma, serum, tricepts",PIMA__STAGING_TABLE_1


---
## 4. Creating a getML Container referencing the Data

The [`Container`](https://getml.com/latest/reference/data/container/#getml.data.Container) holds _all_ data (i.e. populatation, split into train & val sets, _and_ all peripheral tables).
The actual data in a container will be linked to the abstract [**DataModel**](https://getml.com/latest/reference/data/data_model/) during training.

Why a Container is an important construct:
- `Container` is a convenience API that eases handling of relational data for Machine Learning tasks
- The DataModel defines the relational structure; the Container holds references to the actual data.
- No data is duplicated – the Container references the original tables, ensuring efficiency.
- A container allows you to easily pass around relational datasets for training, validation, and testing: E.g. `container.train` holds a [Subset](https://getml.com/latest/reference/data/subset/) that contains the training data for the population and all related peripheral tables.

This separation enhances reproducibility and keeps data handling modular.

In [41]:
container = getml.data.Container(population=pima, split=pima.split)
container.add(**peripheral)

container

population
    subset   name   rows   type
0   train    pima    538   View
1   val      pima    230   View

peripheral
    name        rows   type     
0   age          768   DataFrame
1   bmi          768   DataFrame
2   diastolic    768   DataFrame
3   num_preg     768   DataFrame
4   pedigree     768   DataFrame
5   plasma       768   DataFrame
6   serum        768   DataFrame
7   tricepts     768   DataFrame

---
## 5. Defining the getML Pipeline and Checking the Data

Our [pipeline](https://getml.com/latest/reference/pipeline/) will:
  - Use the DataModel (dm) to understand table relationships,
  - Use [XGBoostClassifier](https://getml.com/latest/reference/predictors/xgboost_classifier/) for prediction.

We do not specify the `feature_learners` parameter for this pipeline.



In [42]:
pipe = getml.pipeline.Pipeline(
    tags=["base_pipeline"],
    data_model=dm,
    predictors=getml.predictors.XGBoostClassifier(), 
)

# .check(...) does an analysis on the pipeline, data model and container
# to see if everything is consistent (time relationships, roles, table joins, etc.).
# We pass the 'train' subset for verification
pipe.check(container.train)

Checking data model...

OK.

---
## 6. Fitting and Scoring the getML Pipeline

In [43]:
# .fit(...) orchestrates learning the features, and training the prediction model.

pipe.fit(container.train, check=False)

  Staging... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00
  XGBoost: Training as predictor... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:01


Trained pipeline.

Time taken: 0:00:01.122228.



Pipeline(data_model='pima',
         feature_learners=[],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['age', 'bmi', 'diastolic', 'num_preg', 'pedigree', 'plasma',
                     'serum', 'tricepts'],
         predictors=['XGBoostClassifier'],
         preprocessors=[],
         share_selected_features=0.5,
         tags=['base_pipeline', 'container-OhbVrp'])

In [44]:
# We now evaluate the pipeline on the validation set:
pipe.score(container.val)

  Staging... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00


,date time,set used,target,accuracy,auc,cross entropy
0,2025-02-20 17:28:45,train,arg2,0.9108,0.9682,0.2754
1,2025-02-20 17:28:47,val,arg2,0.7826,0.8368,0.4777


The high difference between train and validation accuracies indicate overfitted XGBoost classifier. By fine tuning its hyper parameters, the validation accuracy can possibility be increased.

## 7. Failed feature learning attempt

Even if we try to perform feature learning on this simple dataset (by specifying `feature_learners` parameter), we will get an error. getML checks if the data model contains any many-to-one relationships. If not, specifying feature learners does not make sense as feature learning can not performed.


In [56]:
pipe_fl = getml.pipeline.Pipeline(
    tags=["feature_learning_pipeline"],
    data_model=dm,
    predictors=getml.predictors.XGBoostClassifier(),
    feature_learners=getml.feature_learning.FastProp(
        loss_function=getml.feature_learning.loss_functions.CrossEntropyLoss,
    ),
)

pipe_fl.check(container.train)

Checking data model...

OSError: The data model you have passed is not relational (there are no joins in it that aren't many-to-one or one-to-one), yet you have passed relational feature learners.

## Conclusion

This notebook demonstrated that getML can indeed be applied to simple one-to-one relational datasets, which do not require relational learning. We:

1. Loaded and annotated the data.
2. Defined a DataModel to represent the relationships between the tables.
3. Built a pipeline using an XGBoost classifier and evaluated its performance.
4. Observed that the discrepancy between training and validation scores indicating overfitting.
5. Noted that getML checks if feature learning can be performed before trying to perform it

To see getML's feature learning in action for a complex dataset, check out [loans.ipynb](loans.ipynb) which focuses on loan default risk prediction for a Czeck bank. 
